# LineStuffUp: Aligning Two Same-Dimension Volumes

This notebook walks through using the **LineStuffUp** library to align two 3D volumes that share the same dimensions (shape). Because the volumes are the same size, no initial rescaling is needed — you can go straight to interactive alignment.

### Overview of the workflow
1. **Setup** — add LineStuffUp to the Python path and import the library.
2. **Load volumes** — load your fixed (reference) and movable volumes.
3. **Create or load a Graph** — a `Graph` stores nodes (volumes) and edges (transforms between them).
4. **Define an initial transform (optional)** — if you already know an approximate alignment (e.g. a flip or rotation), set it here.
5. **Interactive alignment** — launch the napari GUI to place/adjust alignment manually.
6. **Apply the transform** — warp the movable volume into the fixed space.
7. **Save / reload** — persist the graph and transforms to disk.

---
> **Dependencies:** `linestuffup`, `numpy`, `imageio`, `napari`, `magicgui`, `scipy`, `scikit-image`, `threadpoolctl`, `imageio-ffmpeg`
>
> Install missing packages with:
> ```
> pip install napari magicgui threadpoolctl imageio-ffmpeg scikit-image
> ```

## Cell 1 — Imports and path setup

In [1]:
# =============================================================================
# IMPORTS AND PATH SETUP
# =============================================================================
# If castalign is installed as a package (pip install -e .), no path
# manipulation is needed.  If you are running from a local clone, set
# CASTALIGN_ROOT to the directory that CONTAINS the `castalign` folder
# (i.e. the repo root / castalign-master), NOT the castalign folder itself.

import sys
from pathlib import Path
import numpy as np
import imageio.v2 as imageio

# ---- Point this at the directory that CONTAINS the `castalign` package ----
CASTALIGN_ROOT = Path(r"C:\Users\qy47\Desktop\Neuromics\castalign-master")   # <-- CHANGE THIS if needed

if str(CASTALIGN_ROOT) not in sys.path:
    sys.path.insert(0, str(CASTALIGN_ROOT))
    print(f"Added to path: {CASTALIGN_ROOT.resolve()}")

# Core library
import castalign as lsu
import castalign.gui as lsu_gui
from castalign import ndarray_shifted
import castalign.utils as lsu_utils

# Optional: auto-reload modules during development
%load_ext autoreload
%autoreload 2

print("Imports OK")


Added to path: C:\Users\qy47\Desktop\Neuromics\castalign-master
Imports OK


## Cell 2 — User configuration

Edit the variables in this cell to match your data.

In [2]:
# =============================================================================
# CONFIGURATION  — edit these paths to point at your data
# =============================================================================

# Paths to your two volume files (any format readable by imageio, e.g. .tif, .tiff)
FIXED_PATH   = Path(r'C:\Users\qy47\Desktop\Neuromics\gad2-cre-red-20260211-001\References\combined-tiff.tif')    # The reference/target volume
MOVABLE_PATH = Path(r'C:\Users\qy47\Desktop\Neuromics\gad2-cre red exvivo 260211-001\References\gad2-cre red exvivo 260211 averaged stack.tif')  # The volume you want to align TO the fixed

# Human-readable names used as node labels inside the Graph
FIXED_NAME   = "in vivo stack"
MOVABLE_NAME = "ex vivo stack"

# Where to save / load the alignment Graph (a SQLite .db file)
GRAPH_PATH   = Path(r'C:\Users\qy47\Desktop\Neuromics\Outputs\test 0211 mscarlet volume alignment')

# Set to True if you already have a saved graph and want to load it
LOAD_EXISTING_GRAPH = False

## Cell 3 — Load volumes

LineStuffUp expects 3-D arrays with shape `(Z, Y, X)`.  
`imageio.imread` on a multi-page TIFF returns `(Z, Y, X)` directly.  
If your data come in a different layout, reshape here.

In [3]:
# =============================================================================
# LOAD VOLUMES
# =============================================================================

fixed_raw   = imageio.volread(FIXED_PATH)    # shape (Z, Y, X)
movable_raw = imageio.volread(MOVABLE_PATH)  # shape (Z, Y, X)

# Convert to float32 — required by most linestuffup transforms
fixed   = fixed_raw.astype(np.float32)
movable = movable_raw.astype(np.float32)

print(f"Fixed   shape: {fixed.shape}   dtype: {fixed.dtype}")
print(f"Movable shape: {movable.shape}   dtype: {movable.dtype}")

# Sanity check: the two volumes must have the same dimensions
assert fixed.shape == movable.shape, (
    f"Volume shapes do not match: {fixed.shape} vs {movable.shape}. "
    "This notebook is designed for same-dimension volumes."
)
print("✓ Shapes match — proceeding.")

Fixed   shape: (401, 512, 512)   dtype: float32
Movable shape: (401, 512, 512)   dtype: float32
✓ Shapes match — proceeding.


## Cell 4 — Create (or load) a Graph

A `Graph` is the central data structure in LineStuffUp.  
It stores:
- **Nodes** — named volumes (can also hold compressed image data)
- **Edges** — transforms between pairs of nodes

If you have already run this notebook and saved a graph, set `LOAD_EXISTING_GRAPH = True` in Cell 2 to reload it.

In [4]:
# =============================================================================
# CREATE OR LOAD A GRAPH
# =============================================================================

if LOAD_EXISTING_GRAPH and GRAPH_PATH.exists():
    g = lsu.load(str(GRAPH_PATH))
    print(f"Loaded graph '{g.name}' from {GRAPH_PATH}")
    print(f"  Nodes : {g.nodes}")
    print(f"  Edges : {list(g.edges.keys())}")
else:
    g = lsu.Graph(name="same_dim_alignment")

    # Add nodes.  attach_image=False means the image data is NOT stored
    # inside the graph file — the graph just tracks the node names and
    # transforms.  Set attach_image=True (and pass image=...) if you want
    # the images embedded for portability.
    g.add_node(FIXED_NAME)
    g.add_node(MOVABLE_NAME)

    print(f"Created new graph '{g.name}'")
    print(f"  Nodes: {g.nodes}")

Created new graph 'same_dim_alignment'
  Nodes: ['in vivo stack', 'ex vivo stack']


## Cell 5 — Optional: define an initial transform

Because both volumes have the same dimensions you may not need any initial transform at all — start with `Identity()`.  
However, if you know the movable volume needs a flip or rotation to be roughly oriented the same way as the fixed volume, set it here.

**Common initial transforms:**

| Goal | Code |
|------|------|
| No change (default) | `lsu.Identity()` |
| Translate | `lsu.TranslateFixed(z=10, y=-5, x=0)` |
| Flip along Z | `lsu.FlipFixed(z=True, zthickness=fixed.shape[0])` |
| Flip along X | `lsu.FlipFixed(x=True, xthickness=fixed.shape[2])` |
| Rotate 90° around X | `lsu.TranslateRotateFixed(xrotate=90)` |
| Rotate + flip | `lsu.TranslateRotateFixed(xrotate=90) + lsu.FlipFixed(x=True, xthickness=fixed.shape[2])` |

In [5]:
# =============================================================================
# INITIAL TRANSFORM
# =============================================================================
# For two volumes of identical dimensions the Identity is usually a good start.
# Replace with something more specific if you know the rough orientation differs.

initial_transform = lsu.Identity()

# Example — uncomment if the movable needs a left-right flip:
# initial_transform = lsu.FlipFixed(x=True, xthickness=fixed.shape[2])

# Example — uncomment if the movable needs a rotation AND a flip:
# initial_transform = lsu.TranslateRotateFixed(xrotate=90) + lsu.FlipFixed(x=True, xthickness=fixed.shape[2])

print(f"Initial transform: {initial_transform}")

Initial transform: Identity()


## Cell 6 — Interactive alignment (GUI)

`align_interactive` opens a **napari** window showing both volumes.  
You interact via keyboard prompts printed in the terminal/output cell.

### Key bindings

| Key | Action |
|-----|--------|
| `t` | Translate + rotate (parametric, drag-to-move) |
| `r` | Translate + rotate + rescale (parametric) |
| `T` | Translate + rotate (point-based — click matching landmarks) |
| `R` | Translate + rotate + rescale (point-based) |
| `V` | Non-linear triangulation (point-based, for fine warping) |
| `f` | Flip along Z |
| `e` | Edit the previous parametric transform via the side panel |
| `z` | Remove the most recently added transform |
| `u` | Undo the last change |
| `s` | Save transform to the graph (in memory) |
| `S` | Save transform to the graph AND write to disk |
| `q` | Quit the GUI and return the current transform |

The function **blocks** until you press `q`.  
The returned `t` is the fitted `Transform` object.

In [ ]:
# =============================================================================
# INTERACTIVE ALIGNMENT
# =============================================================================
# This will open a napari window.  Follow the on-screen prompts.
# The fixed volume is shown in white/grey; the movable is shown in colour.
# Manipulate the movable until it overlaps the fixed, then press 'q'.

t = lsu_gui.align_interactive(
    nodes_movable = movable,        # The volume to warp
    nodes_fixed   = fixed,          # The reference volume
    transform     = initial_transform,  # Starting point (can be Identity)
    graph         = None,           # Pass `g` here if you want in-GUI save (s/S)
)

print()
print("Fitted transform:")
print(t)

Current transform is: Identity()

Please choose an option:

Parametric transforms
---------------------
t: Translate and rotate
r: Translate, rotate, and rescale

Point-based transforms
----------------------
T: Translate and rotate
R: Translate, rotate, and rescale
P: Translate, rotate, and rescale along a plane
N: Nonlinear projected triangulation
V: Nonlinear 3D triangulation

Modify last transform
---------------------
e: edit previous transform
z: remove the previous transform
x_: Extend previous point-based transform with a different point-based transform
c_: Convert previous point-based transform to a different point-based transform
(where _ is the letter key for any point based transform)

Other
-----
v: view
f: flip along z axis
u: revert most recent change
q: quit

Setting default params
TranslateRotate(points_start=[[190.0, 332.0, 309.0], [180.0, 286.0, 221.0], [96.0, 206.0, 351.0], [215.0, 243.0, 238.0], [223.0, 259.0, 221.0], [245.0, 319.0, 319.0]], points_end=[[210.0, 324

## Cell 7 — Store the transform in the graph

The graph records the transform as a directed edge from `MOVABLE_NAME` to `FIXED_NAME`.  
This edge can later be traversed to warp any data between the two spaces.

In [ ]:
# =============================================================================
# ADD / UPDATE EDGE IN GRAPH
# =============================================================================

if (MOVABLE_NAME, FIXED_NAME) in g:     # Edge already exists → update it
    print("Edge already exists — updating.")
    g.add_edge(MOVABLE_NAME, FIXED_NAME, t, update=True)
else:
    g.add_edge(MOVABLE_NAME, FIXED_NAME, t)
    print("Edge added.")

print(f"  {MOVABLE_NAME} → {FIXED_NAME} : {t}")

Edge added.
  ex vivo stack → in vivo stack : TranslateRotateFixed(z=0.0, y=0.0, x=0.0, zrotate=0.0, yrotate=0.0, xrotate=0.0, invert=False)


## Cell 8 — Apply the transform and inspect the result

`transform_image` warps `movable` into the coordinate space of `fixed`.  
Because both volumes have the same shape, we pass `output_size` equal to `fixed.shape` so the output is guaranteed to be the same size.

In [ ]:
# =============================================================================
# APPLY THE TRANSFORM
# =============================================================================

# output_size forces the result to exactly match the fixed volume dimensions.
# This is only valid because the two volumes share the same shape.
warped = t.transform_image(
    movable,
    output_size = list(fixed.shape),   # [(0, Z), (0, Y), (0, X)] bounding box
)

print(f"Warped shape : {np.asarray(warped).shape}")
print(f"Fixed  shape : {fixed.shape}")

# Quick pixel-level check: how well do the two volumes overlap?
warped_arr = np.asarray(warped, dtype=np.float32)
corr = np.corrcoef(fixed.ravel(), warped_arr.ravel())[0, 1]
print(f"Pearson correlation (fixed vs warped): {corr:.4f}  "
      f"(1.0 = perfect, 0 = none)")

Warped shape : (401, 512, 512)
Fixed  shape : (401, 512, 512)
Pearson correlation (fixed vs warped): 0.0464  (1.0 = perfect, 0 = none)


## Cell 9 — Visual inspection in napari

In [ ]:
# =============================================================================
# VISUAL INSPECTION
# =============================================================================
# Opens a napari viewer with three layers so you can compare side-by-side
# or toggle layers on/off.

import napari

viewer = napari.Viewer(title="Alignment check")
viewer.add_image(fixed,       name="fixed",   colormap="gray",  blending="additive", opacity=0.7)
viewer.add_image(warped_arr,  name="warped",  colormap="green", blending="additive", opacity=0.5)
viewer.add_image(movable,     name="movable (original)", colormap="red", blending="additive", opacity=0.3, visible=False)

napari.run()   # Blocks until the viewer window is closed

## Cell 10 — Save the graph and the warped volume

In [ ]:
# =============================================================================
# SAVE GRAPH (transforms)
# =============================================================================
# The graph is saved as a SQLite .db file.  You can reload it later with
# lsu.load(str(GRAPH_PATH)).

if GRAPH_PATH.exists():
    # Graph.save() will not overwrite by default; delete first.
    GRAPH_PATH.unlink()

g.save(str(GRAPH_PATH))
print(f"Graph saved to: {GRAPH_PATH.resolve()}")

# =============================================================================
# SAVE WARPED VOLUME
# =============================================================================
WARPED_PATH = Path("warped_volume.tif")

# Convert back to the same dtype as the input if needed
save_arr = warped_arr.astype(fixed_raw.dtype)
imageio.mimwrite(str(WARPED_PATH), save_arr)  # mimwrite handles multi-page TIFFs
print(f"Warped volume saved to: {WARPED_PATH.resolve()}")

PermissionError: [WinError 5] Access is denied: 'C:\\Users\\qy47\\Desktop\\Neuromics\\Outputs\\test 0211 mscarlet volume alignment'

## Cell 11 — Reload and verify the saved graph

This cell shows how to reload the graph in a fresh session and reconstruct the transform.

In [ ]:
# =============================================================================
# RELOAD GRAPH AND RE-APPLY TRANSFORM
# =============================================================================

g_reload = lsu.load(str(GRAPH_PATH))
t_reload = g_reload.get_transform(MOVABLE_NAME, FIXED_NAME)

print(f"Reloaded graph: '{g_reload.name}'")
print(f"  Nodes : {g_reload.nodes}")
print(f"  Transform {MOVABLE_NAME}→{FIXED_NAME} : {t_reload}")

# Verify it produces the same output
warped_check = np.asarray(t_reload.transform_image(movable, output_size=list(fixed.shape)), dtype=np.float32)
max_diff = np.max(np.abs(warped_check - warped_arr))
print(f"  Max absolute difference after reload: {max_diff:.6f}  (should be ~0)")

---
## Tips and troubleshooting

**The GUI blocks execution** — that is expected. Press `q` in the napari terminal prompt to return the transform and continue.

**Point-based transforms (T, R, V)** require you to click matching landmarks in both the fixed and movable layers inside napari. Add at least 4 landmark pairs before pressing `q`.

**Same-shape assumption** — `output_size=list(fixed.shape)` in `transform_image` is only correct when both volumes share the same voxel grid. If the volumes were to differ in resolution or extent in a future use-case, remove `output_size` to let LineStuffUp compute the minimal bounding box automatically.

**Label images** — if one of your volumes is a segmentation/label image, pass `labels=True` to `transform_image` to disable interpolation:
```python
warped_labels = t.transform_image(movable_labels, output_size=list(fixed.shape), labels=True)
```

**Composing transforms** — transforms can be chained with `+`:
```python
composed = lsu.FlipFixed(x=True, xthickness=fixed.shape[2]) + lsu.TranslateFixed(z=5)
```

**Inverting a transform** — to warp the fixed volume into movable space:
```python
warped_inverse = t.invert().transform_image(fixed)
```